In [1]:
import os
import sys
import django

# Add project root to path and configure Django
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'settings.settings')
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'
django.setup()

In [2]:
import pandas as pd
from jobs.models import Job, ExtendedJob

## 1. Load Raw Data

In [3]:
# Query jobs that have extended job data using select_related for efficiency
jobs_with_extended = ExtendedJob.objects.select_related('job').values(
    'job__title',
    'job__company',
    'job__location',
    'job__salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
)

# Convert to DataFrame
df = pd.DataFrame(list(jobs_with_extended))

# Rename columns to remove the 'job__' prefix
df.columns = [
    'title',
    'company',
    'location',
    'salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
]

df

,title,company,location,salary,bachelor_required,master_required,phd_required,tech_stack,min_experience_years,us_only,employment_type,medical_insurance
0,Service Desk Support Analyst III - Kelsey - Se...,Optum,", US, TXPearland",$28.94 - $51.63 an hour,True,False,False,"microsoft office, windows desktop, windows ope...",4,True,full-time,True
1,Principal Enterprise Architect (Remote),IQVIA,", US, NJWayne","$118,100 - $328,800 a year",False,False,False,"Kafka, Apigee, Azure API Management, Kong, Azu...",15,True,full-time,True
2,Senior Technical Program Manager (ST PM) (15.35),"OCT Consulting, LLC",", US, DCWashington","$175,000 - $250,000 a year",True,False,False,"AWS, cybersecurity, cloud operations, Agile, I...",8,True,full-time,True
3,Cloud Cybersecurity Manager (CCM) (15.35),"OCT Consulting, LLC",", US, DCWashington","$150,000 - $225,000 a year",True,False,False,"aws, nist, dod, cybersecurity, cloud security",8,True,full-time,True
4,Cloud Architect and Infrastructure Lead (CAIL)...,"OCT Consulting, LLC",", US, DCWashington","$150,000 - $200,000 a year",True,False,False,"AWS,DevSecOps,Agile",6,True,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1907,"Senior Staff Software Engineer, Backend (Consu...",Coinbase,", USRemote","$246,500 - $290,000 a year",False,False,False,"go,temporal,kubernetes,mongodb",10,False,full-time,True
1908,"Staff Software Engineer, Backend (Consumer - T...",Coinbase,", USRemote","$218,025 - $256,500 a year",True,False,False,"golang,clickhouse,kafka,redis,mongodb,blockcha...",8,False,full-time,True
1909,Engineering Manager (Consumer - Growth),Coinbase,", USRemote","$218,025 - $256,500 a year",False,False,False,NaN,7,False,full-time,True
1910,"Senior Staff Software Engineer, Backend (Consu...",Coinbase,", USRemote","$253,895 - $298,700 a year",False,False,False,NaN,0,False,full-time,True


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   title                 1912 non-null   str  
 1   company               1912 non-null   str  
 2   location              1912 non-null   str  
 3   salary                1912 non-null   str  
 4   bachelor_required     1912 non-null   bool 
 5   master_required       1912 non-null   bool 
 6   phd_required          1912 non-null   bool 
 7   tech_stack            1662 non-null   str  
 8   min_experience_years  1912 non-null   int64
 9   us_only               1912 non-null   bool 
 10  employment_type       1773 non-null   str  
 11  medical_insurance     1912 non-null   bool 
dtypes: bool(5), int64(1), str(6)
memory usage: 114.0 KB


## 2. Clean Data

### 2.1 Clean: Salary

In [8]:
print(df['salary'].sample(30).to_string())

1623    $170,000 - $720,000 a year
1173    $175,000 - $230,000 a year
1599    $107,744 - $140,300 a year
1443    $115,000 - $127,000 a year
1059     $88,300 - $111,200 a year
1423    $163,600 - $303,100 a year
166     $225,000 - $250,000 a year
42                  $25,000 a year
549     $140,400 - $372,300 a year
1655      $65,000 - $85,000 a year
1394    $118,450 - $260,590 a year
724     $178,200 - $297,000 a year
1331    $140,000 - $175,000 a year
711                    $25 an hour
413     $138,900 - $191,000 a year
1488     $90,000 - $134,000 a year
1373    $120,000 - $160,000 a year
872     $170,000 - $220,000 a year
1200     $90,000 - $123,000 a year
1664    $266,000 - $345,700 a year
1867             $60 - $70 an hour
579       $70,000 - $75,000 a year
268      $85,093 - $103,500 a year
681     $190,000 - $220,000 a year
1449    $180,000 - $210,000 a year
909     $150,000 - $175,000 a year
1247     $60,000 - $150,000 a year
1733    $126,700 - $215,400 a year
80       $96,100 - $

In [5]:
import re

def clean_salary_to_hourly(salary_str):
    """
    Convert salary string to hourly rate (float).
    - Removes $ and , symbols
    - Averages MIN-MAX ranges
    - Converts monthly (200-40k) and annual (>40k) to hourly
    """
    if pd.isna(salary_str) or not salary_str:
        return None
    
    # Remove $ and , symbols
    cleaned = salary_str.replace('$', '').replace(',', '')
    
    # Extract all numbers (including decimals)
    numbers = re.findall(r'[\d.]+', cleaned)
    
    if not numbers:
        return None
    
    # Convert to floats
    values = [float(n) for n in numbers]
    
    # Average if range (MIN-MAX)
    salary = sum(values) / len(values)
    
    # Convert to hourly based on value thresholds
    if salary < 200:
        # Already hourly
        hourly = salary
    elif salary < 40000:
        # Monthly -> divide by 173.33
        hourly = salary / 173.33
    else:
        # Annual -> divide by 2080
        hourly = salary / 2080
    
    return round(hourly, 2)

# Apply the cleaning function
df['salary_hourly'] = df['salary'].apply(clean_salary_to_hourly)

# Show sample of original vs cleaned
df[['salary', 'salary_hourly']].head(10)

,salary,salary_hourly
0,$28.94 - $51.63 an hour,40.29
1,"$118,100 - $328,800 a year",107.43
2,"$175,000 - $250,000 a year",102.16
3,"$150,000 - $225,000 a year",90.14
4,"$150,000 - $200,000 a year",84.13
5,"$120,001 - $160,000 a year",67.31
6,"$140,000 - $165,000 a year",73.32
7,"$111,000 - $156,500 a year",64.30
8,"$105,000 - $125,000 a year",55.29
9,"$45,000 - $110,000 a year",37.26


### 2.2 Clean: Employment type

In [10]:
df['employment_type'].unique()

<StringArray>
[                   'full-time',                            nan,
                    'part-time',                     'contract',
           'full-time,contract',                   'internship',
          'full-time,part-time', 'full-time,part-time,contract']
Length: 8, dtype: str

In [11]:
# Clean employment_type: keep only the first value if multiple exist
df['employment_type'] = df['employment_type'].str.split(',').str[0]

df['employment_type'].unique()

array(['full-time', nan, 'part-time', 'contract', 'internship'],
      dtype=object)

### 2.3 Clean: Location

In [12]:
df['location'].unique()

<StringArray>
[     ', US, TXPearland',         ', US, NJWayne',    ', US, DCWashington',
       ', US, TXEl Paso',        ', US, TXIrving',        ', US, TXAustin',
  ', US, CAPort Hueneme',  ', US, COFort Collins',       ', US, VAFairfax',
       ', US, WAPullman',
 ...
       ', US, MIZeeland',          ', US, MSnull',     ', US, FLLake Mary',
    ', US, MASomerville',    ', US, OHCincinnati', ', US, OKOklahoma City',
   ', US, MNMaple Grove', ', US, ILDowners Grove',     ', US, NYGreenwich',
    ', US, ALBirmingham']
Length: 296, dtype: str

In [13]:
print(df['location'].sample(30).to_string())

0           , US, TXPearland
1576          , US, TXAustin
103               , USRemote
990       , US, CAPleasanton
1704              , USRemote
1666              , USRemote
1269         , US, ILChicago
1201         , US, NYBuffalo
734                 , USnull
708           , US, VAReston
1629              , USRemote
259               , USRemote
1697              , USRemote
1032              , USRemote
397               , USRemote
564               , USRemote
1911              , USRemote
1706              , USRemote
1596            , US, INnull
9       , US, COFort Collins
1327            , US, MTnull
61                , USRemote
896               , USRemote
1622              , USRemote
1543              , USRemote
362           , US, NYAlbany
1597              , USRemote
857           , US, OHToledo
24        , US, DCWashington
608       , US, NHManchester


In [14]:
def clean_location(loc_str):
    """
    Clean location to format: "US" or "US, Remote" or "US, STATE"
    
    Input formats:
    - ', US, TXPearland' -> 'US, TX'
    - ', USRemote' -> 'US, Remote'
    - ', US, MSnull' -> 'US, MS'
    - ', USnull' -> 'US'
    """
    if pd.isna(loc_str) or not loc_str:
        return 'US'
    
    # Remove leading comma and spaces
    cleaned = loc_str.strip().lstrip(',').strip()
    
    # Check for Remote
    if 'Remote' in cleaned:
        return 'US, Remote'
    
    # Check for just "USnull" (no state)
    if cleaned == 'USnull':
        return 'US'
    
    # Pattern: "US, STATEcity" - extract state code (2 letters after "US, ")
    if ', ' in cleaned:
        parts = cleaned.split(', ')
        if len(parts) >= 2:
            state_city = parts[1]
            # Extract first 2 characters as state code
            state = state_city[:2].upper()
            return f'US, {state}'
    
    return 'US'

# Apply the cleaning function
df['location'] = df['location'].apply(clean_location)

# Check unique values
df['location'].unique()

<StringArray>
[    'US, TX',     'US, NJ',     'US, DC',     'US, CA',     'US, CO',
     'US, VA',     'US, WA',     'US, MN',     'US, UT',     'US, FL',
     'US, GA',     'US, MA',     'US, IL',     'US, OH',     'US, WV',
     'US, AZ',     'US, SC',     'US, NC', 'US, Remote',         'US',
     'US, IN',     'US, MD',     'US, MI',     'US, NY',     'US, KY',
     'US, ME',     'US, NV',     'US, NM',     'US, PA',     'US, OR',
     'US, NH',     'US, CT',     'US, MO',     'US, WI',     'US, OK',
     'US, TN',     'US, KS',     'US, LA',     'US, RI',     'US, NE',
     'US, ND',     'US, IA',     'US, MT',     'US, VT',     'US, ID',
     'US, AL',     'US, WY',     'US, MS']
Length: 48, dtype: str

## 2.4 Clean: Title

In [15]:
print(df['title'].sample(30).to_string())

442                               Senior Digital Designer
1615      Data Engineer - Data Foundation Enablement Team
1195                           Senior Engineering Manager
1670                                Senior Data Scientist
927           Security Solution Engineer - Email Security
1408                                        Data Engineer
786     Experienced Infrastructure Engineer - (100% Re...
1647    Customer Care Specialist - Contract (open to r...
1751                              Staff Software Engineer
607                      Principal Data Solutions Analyst
968                        Head of Solutions Architecture
1903                                    VP of Data and AI
605                 Sr. Director of Business Applications
1039              Associate Software Engineer (US Remote)
325                                Senior Product Manager
977                  Supply Tech Business Systems Partner
917                                  Intern, Scrum Master
1495        So

In [ ]:
def categorize_title(title):
    """
    Categorize job title into one of 13 categories.
    Order matters - more specific matches come first.

    Categories:
    {
        "SWE": "Software Engineer, Full Stack, Frontend, Backend, Developer",
        "SSWE": "Staff Engineer, Principal Engineer, Software Architect",
        "C": "C-level (CTO, COO, CEO, VP, Head of, etc.)",
        "MLE": "Machine Learning Engineer, Data Scientist, AI Engineer",
        "DevOps": "DevOps, SRE, Infrastructure, Security, Cloud Engineer/Architect",
        "DA": "Data Analyst, Business Analyst",
        "DE": "Data Engineer, ETL Developer, Data Architect",
        "PM": "Product/Project/Program Manager, Scrum Master",
        "EM": "Engineering Manager, Director of Engineering",
        "TL": "Tech Lead, Team Lead, Lead Developer",
        "UI": "UI/UX Designer, Product Designer",
        "QA": "QA Engineer, SDET, Test Engineer",
        "HD": "Help Desk like jobs",
        "Other": "Everything else"
    }
    """
    if pd.isna(title) or not title:
        return 'Other'
    
    title_lower = title.lower()
    
    # C-Level executives (check early - these are specific)
    c_patterns = ['cto', 'coo', 'ceo', 'cfo', 'cio', 'ciso', 'cpo', 
                  'chief ', 'vp ', 'vp,', 'vice president', 
                  ' vp', 'svp', 'evp', 'head of']
    if any(p in title_lower for p in c_patterns):
        return 'C'
    
    # Engineering Manager (check before Software Engineer)
    em_patterns = ['engineering manager', 'engineer manager', 'software manager',
                   'development manager', 'director of engineering', 
                   'director, engineering', 'director of software']
    if any(p in title_lower for p in em_patterns):
        return 'EM'
    
    # Senior Staff / Principal (check before TL and SWE)
    sswe_patterns = ['staff engineer', 'staff software', 'senior staff',
                     'principal engineer', 'principal software',
                     'software architect', 'application architect',
                     'distinguished engineer']
    if any(p in title_lower for p in sswe_patterns):
        return 'SSWE'
    
    # Tech Lead (check before Software Engineer)
    tl_patterns = ['tech lead', 'technical lead', 'team lead', 'lead engineer',
                   'lead software', 'lead developer']
    if any(p in title_lower for p in tl_patterns):
        return 'TL'
    
    # Product/Project Manager
    pm_patterns = ['product manager', 'project manager', 'program manager',
                   'product owner', 'scrum master', 'agile coach',
                   'delivery manager', 'release manager', 'technical program']
    if any(p in title_lower for p in pm_patterns):
        return 'PM'
    
    # MLE - Machine Learning / Data Science (check before DA/DE)
    mle_patterns = ['machine learning', 'ml engineer', 'data scientist', 
                    'ai engineer', 'deep learning', 'nlp engineer',
                    'computer vision', 'research scientist', 'applied scientist',
                    'mlops', 'ml ops']
    if any(p in title_lower for p in mle_patterns):
        return 'MLE'
    
    # Data Engineer (check before DA and Software Engineer)
    de_patterns = ['data engineer', 'etl developer', 'data architect',
                   'data platform', 'data infrastructure', 'analytics engineer',
                   'bi developer', 'warehouse engineer', 'dataops']
    if any(p in title_lower for p in de_patterns):
        return 'DE'
    
    # Data Analyst
    da_patterns = ['data analyst', 'business analyst', 'analytics analyst',
                   'bi analyst', 'business intelligence analyst', 'reporting analyst',
                   'insights analyst', 'data analytics', 'quantitative analyst']
    if any(p in title_lower for p in da_patterns):
        return 'DA'
    
    # DevOps / Infrastructure / Security / Cloud
    devops_patterns = ['devops', 'devsecops', 'sre', 'site reliability',
                       'infrastructure', 'platform engineer', 'cloud engineer',
                       'security engineer', 'cybersecurity', 'infosec',
                       'network engineer', 'systems engineer', 'system engineer',
                       'cloud architect', 'solutions architect', 'secops',
                       'kubernetes', 'aws engineer', 'azure engineer',
                       'gcp engineer']
    if any(p in title_lower for p in devops_patterns):
        return 'DevOps'
    
    # UI/UX Designer
    ui_patterns = ['ui ', 'ux ', 'ui/', 'ux/', 'user interface', 
                   'user experience', 'product designer', 'visual designer',
                   'interaction designer', 'design system']
    if any(p in title_lower for p in ui_patterns):
        return 'UI'
    
    # QA
    qa_patterns = ['qa ', 'qae', 'quality assurance', 'test engineer',
                   'sdet', 'automation engineer', 'quality engineer',
                   'test automation', 'testing engineer', 'tester']
    if any(p in title_lower for p in qa_patterns):
        return 'QA'
    
    # Software Engineer (broad catch for developers)
    swe_patterns = ['software engineer', 'software developer', 'developer',
                    'full stack', 'fullstack', 'frontend', 'front end', 'front-end',
                    'backend', 'back end', 'back-end', 'web developer',
                    'mobile developer', 'ios developer', 'android developer',
                    'java developer', 'python developer', '.net developer',
                    'react developer', 'node developer', 'golang',
                    'application developer', 'software development',
                    'programmer', 'coder', 'engineer, software']
    if any(p in title_lower for p in swe_patterns):
        return 'SWE'
    
    return 'Other'

# Apply categorization
df['title_category'] = df['title'].apply(categorize_title)

# Show distribution
print("Title Category Distribution:")
print(df['title_category'].value_counts())
print(f"\nTotal categories: {df['title_category'].nunique()}")

Title Category Distribution:
title_category
Other     618
SWE       357
PM        223
DevOps    166
C         151
MLE        87
DE         73
SSWE       67
DA         51
UI         35
EM         34
QA         28
TL         22
Name: count, dtype: int64

Total categories: 13


In [21]:
df[['title', 'title_category']].head(30)

,title,title_category
0,Service Desk Support Analyst III - Kelsey - Se...,Other
1,Principal Enterprise Architect (Remote),Other
2,Senior Technical Program Manager (ST PM) (15.35),PM
3,Cloud Cybersecurity Manager (CCM) (15.35),DevOps
4,Cloud Architect and Infrastructure Lead (CAIL)...,DevOps
5,Python Web App Developer,SWE
6,Sr. Gen AI Architect,Other
7,REMOTE-Pension Domain – IT Systems & Solution ...,Other
8,#2612 Senior Test Engineer (TE40),QA
9,CLOUD PRODUCT ENGINEERING TESTING - L3,Other
